# 19.3 同一題的不同版本，怎麼避免跑進兩份考卷？


同一張包的原圖，可以縮放成單物件題，也可以和短靴配成左右關係題。如果原圖進了訓練，縮放版卻進最後考卷，模型就可能靠見過的素材答題。「家族」在這裡指同一來源或情境及它的衍生題；先決定家族去哪份資料，再製作不同問法與版本。

下圖用包的來源A示意：單物件、配對與交換位置的題目都留在訓練份。配對只能使用已分到同一份的兩個來源；圖中的短靴也來自訓練份，不能拿驗證份的短靴和訓練份的包組成訓練題。驗證和最後考卷使用各自另留的來源，而不是把A的不同版本隨機拆散。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p6-19-start-splits.svg")))

本章的固定資料分成以下三份。題數是對話記錄數：同一張圖或錄音可能產生多筆記錄，因此題數不是原始素材數。

| 資料份 | 對話記錄數 | 用途 |
| --- | ---: | --- |
| train：訓練 | 28,876 | 參與梯度與參數更新 |
| validation：驗證 | 2,435 | 檢查學習狀況、選定檢查點 |
| test：最後考卷 | 3,734 | 定版後核對成品，不再用來選權重 |

不同材料需要不同的家族規則；只要求每筆記錄的ID不同還不夠。

| 材料 | 一起留在同一份的內容 |
| --- | --- |
| 文字與對話 | 同一改述家族、客服情境樹，以及它的格式、範圍與歷史版本 |
| 計算工具 | 同一組數字及交換順序、不同運算、工具可用性與結果回填題；問法家族也隔離 |
| 商品圖 | 先按原始商品ID與原始像素去重切分；只在同一份內配對，再製作單物件、方向與換位題 |
| 字卡 | 2–4字組合按字串隔離；同一來源家族的區域問題及訓練增強版本留在同一份 |
| 錄音 | 同一來源錄音、重複或近重複語句群組，以及由它衍生的格式、續聊與增強題 |

字卡考的是已知字元的新組合與版面。12個單字本來就共同出現在學習及單字檢查中，不能稱為新字測試；V2訓練已包含 Sans 和 Serif，考卷中的 Serif 也不能稱為陌生字型。提供的讀字區域是一個連續1–4格框，模型學的是框內辨識，沒有另外學文字偵測。

錄音則有62份訓練、15份驗證和30份最後素材。這些數字數的是**錄音**；來源沒有說話者ID，不能換寫成62人、15人、30人，或宣稱最後考卷的人從未出現在訓練中。來源轉寫可供作者整理重複群組及檢查文字題洩漏，推論輸入仍只有實際聲音和公開對話，不附逐字稿、意圖標籤或當前標準答案。

資料隔離完成後，還要看答案分佈和成對題：例如只回答「包」能猜中多少商品題，同圖換位置是否真的改變答案。零家族交集排除了指定來源的跨份重複，不能單獨證明模型使用了材料。這些檢查會在感知和最終驗收時與模型回答一起看。

<details>
<summary>補充：怎樣把資料版本固定下來？</summary>

[V2 manifest](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/v2-manifest.json)列出12個記錄檔與全部素材的路徑、大小和完整SHA-256指紋。訓練 wrapper 啟動時會核對這些檔案；不同階段沿用同一份固定資料。V2的文字、工具、字卡與聲音增強只追加訓練素材，原有驗證和最後記錄及媒體保持原檔內容。

`group_id` 是記錄裡的家族標記，[資料讀取器](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/tiny_perceptron/selftrained/dataset.py)會拒絕同一標記跨份；商品來源像素、字串與錄音群組的隔離則由各自的資料準備程式另外檢查，不能只依賴一個標記。固定資料的取得與逐檔驗證見[訓練操作頁](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/TRAINING.md#安裝與固定資料)，來源與授權見[V2資料說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/V2-DATA-NOTICE.md)。

</details>
